In [1]:
##Simple GENAI APP using LangChain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")  # make sure this exists too
os.environ['LANGSMITH_API_KEY'] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ['LANGSMITH_PROJECT']=os.getenv("LANGSMITH_PROJECT")

In [4]:
#Data Ingestion=>Loading the data
##From website we will scrape the data
from langchain_community.document_loaders import WebBaseLoader

In [6]:
loader=WebBaseLoader("https://react.dev/learn")
docs=loader.load()
docs

[Document(metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='Quick Start – ReactReactv19.2Search⌘CtrlKLearnReferenceCommunityBlogGET STARTEDQuick Start Tutorial: Tic-Tac-Toe Thinking in React Installation Creating a React App Build a React App from Scratch Add React to an Existing Project Setup Editor Setup Using TypeScript React Developer Tools React Compiler Introduction Installation Incremental Adoption Debugging and Troubleshooting LEARN REACTDescribing the UI Your First Component Importing and Exporting Components Writing Markup with JSX JavaScript in JSX with Curly Braces Passing Props to a Component Conditional Rendering Rendering Lists Keeping Components Pure Your UI as a Tree Adding Interactivity Responding to Events State: A Component\'s Memory Render and Commit State as a Snapshot Queueing a Series of State Updates Updating Objects in State Updating Arrays in State Managing State Reacting to Input with State Choosi

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)

In [9]:
documents=splitter.split_documents(docs)

In [10]:
documents

[Document(metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='Quick Start – ReactReactv19.2Search⌘CtrlKLearnReferenceCommunityBlogGET STARTEDQuick Start Tutorial: Tic-Tac-Toe Thinking in React Installation Creating a React App Build a React App from Scratch Add'),
 Document(metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='a React App Build a React App from Scratch Add React to an Existing Project Setup Editor Setup Using TypeScript React Developer Tools React Compiler Introduction Installation Incremental Adoption'),
 Document(metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='Introduction Installation Incremental Adoption Debugging and Troubleshooting LEARN REACTDescribing the UI Your First Component Importing and Exporting Components Writing Markup with JSX JavaScript in'),
 Document(metadata={'source':

In [11]:
##Coverting into vectors
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [13]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [15]:
result=vectorstoredb.similarity_search("what is react?")
result

[Document(id='c7c8f15b-bf54-4600-8948-d48622a4061c', metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='React apps are made out of components. A component is a piece of the UI (user interface) that has its own logic and appearance. A component can be as small as a button, or as large as an entire page.'),
 Document(id='765f01de-3d63-4ba9-9480-c7fbb9fac776', metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='Not Need an Effect Lifecycle of Reactive Effects Separating Events from Effects Removing Effect Dependencies Reusing Logic with Custom Hooks Learn ReactQuick StartWelcome to the React documentation!'),
 Document(id='9fb54897-e5b8-4e29-b20c-8440152f2734', metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='StartWelcome to the React documentation! This page will give you an introduction to 80% of the React

In [16]:
query="A component can be as small as a button"
result=vectorstoredb.similarity_search(query)

In [18]:
result[0].page_content

'React apps are made out of components. A component is a piece of the UI (user interface) that has its own logic and appearance. A component can be as small as a button, or as large as an entire page.'

In [25]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [ ]:
##Retrieval Chain,Document Chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
    Answer the following question based on the the provided context:
    <context>
    {context}
    </context> 
    """
)
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the the provided context:\n    <context>\n    {context}\n    </context> \n    '), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_c

In [29]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"A component can be as small as a button",
    "context":[Document(page_content="A component can be as small as a button, or as large as an entire page.")]
})

'What is an example of the possible size of a component?  \n\nAn example of the possible size of a component could be a button, representing a small component, or an entire page, representing a large component.'

In [31]:
##Retriever=>it can be considered as interface==> if anyone asks any input then retriever will be the way 
# to get the data from vectordb where we don't need to do similarity search
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x13168bd10>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the the provided context:\n    <context>\n    {context}\n    </context> \n    '), additional_kwargs={})])
     

In [35]:
##Get the response from the LLM
response=retrieval_chain.invoke({"input":"A component can be as small as a button"})
response

{'input': 'A component can be as small as a button',
 'context': [Document(id='c7c8f15b-bf54-4600-8948-d48622a4061c', metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='React apps are made out of components. A component is a piece of the UI (user interface) that has its own logic and appearance. A component can be as small as a button, or as large as an entire page.'),
  Document(id='da0151c4-013a-4c7d-bef5-3db8c622e2cb', metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='Sharing data between components \nIn the previous example, each MyButton had its own independent count, and when each button was clicked, only the count for the button clicked changed:'),
  Document(id='a2cdeff1-5905-4551-9971-57f21c2c6532', metadata={'source': 'https://react.dev/learn', 'title': 'Quick Start – React', 'language': 'en'}, page_content='Creating and nesting components'),
  Document(i